# Merchant Segmentation, Top-10 per Segment & Top-100 Overall Analysis

This notebook segments BNPL merchants into **5 retail industry categories** and evaluates them on:
1. **Financial Scale (GMV)**: Total transaction dollar value processed.
2. **Customer Reach & Loyalty**: Distinct consumer reach and transaction volume.
3. **Average Order Value (AOV)**: Average transaction ticket size.
4. **Risk & Fraud Exposure**: Predicted fraud rate and expected fraud loss.
5. **Risk-Adjusted Merchant Performance Index (MPI)**: Normalized composite index balancing revenue scale against safety.

### Outputs Generated:
- `top_10_merchants_per_segment.csv`: Top 10 merchants per retail segment (50 merchants total).
- `top_100_merchants_overall.csv`: Top 100 best merchants overall across all retail categories.
- `all_merchants_segmented_ranked.csv`: Full universe of 4,026 merchants with both segment and overall rankings.

In [ ]:
import gc
from pathlib import Path
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
import seaborn as sns

# Path configuration (works from root or notebooks/)
PROJECT_ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
TABLES_DIR = PROJECT_ROOT / "data" / "tables"
PART1_DIR = TABLES_DIR / "part_1"
CURATED_DIR_1 = PROJECT_ROOT / "data" / "curated" / "curated_transactions"
CURATED_DIR_2 = PROJECT_ROOT / "data" / "curated"
CURATED_DIR = CURATED_DIR_1 if CURATED_DIR_1.exists() and list(CURATED_DIR_1.glob('*.parquet')) else CURATED_DIR_2
OUTPUT_DIR = TABLES_DIR

# File paths
MERCHANTS_PATH = PART1_DIR / "tbl_merchants.parquet"
PREDICTIONS_PARQUET = TABLES_DIR / "part_2_4_fraud_probability_predictions.parquet"
PREDICTIONS_CSV = TABLES_DIR / "part_2_4_fraud_probability_predictions_with_flag.csv"


In [ ]:
# ------------------------------------------------------------
# 1. Load Merchants & Define 5 Strategic Industry Segments
# ------------------------------------------------------------
merchants = pd.read_parquet(MERCHANTS_PATH)
if "merchant_abn" not in merchants.columns:
    merchants = merchants.reset_index()

# Robust tag-based segmentation function
def assign_merchant_segment(tags_str):
    """
    Classify merchants into 5 strategic retail segments based on industry tags.
    """
    if not isinstance(tags_str, str):
        return "5. Luxury, Specialty & Other"
    
    t = tags_str.lower()
    
    # 1. Fashion & Apparel
    if any(k in t for k in ["cloth", "apparel", "shoe", "fashion", "wear", "footwear", "boutique", "dress", "sportswear"]):
        return "1. Fashion & Apparel"
    
    # 2. Electronics & Technology
    elif any(k in t for k in ["electr", "tech", "comput", "digital", "gadget", "phone", "audio", "appliance", "cable", "satellite", "telecom"]):
        return "2. Electronics & Technology"
    
    # 3. Home, Living & Furniture
    elif any(k in t for k in ["furniture", "home", "interior", "hardware", "garden", "decor", "kitchen", "bedding", "lighting", "nursery"]):
        return "3. Home, Living & Furniture"
    
    # 4. Health, Beauty & Wellness
    elif any(k in t for k in ["health", "beauty", "cosmetic", "pharmacy", "wellness", "fitness", "medical", "optometrist", "salon", "spa"]):
        return "4. Health, Beauty & Wellness"
    
    # 5. Luxury, Jewelry & Specialty
    else:
        return "5. Luxury, Jewelry & Specialty"

merchants["segment"] = merchants["tags"].apply(assign_merchant_segment)

print(f"Total merchants loaded: {len(merchants):,}")
print("\nMerchants per segment:")
print(merchants["segment"].value_counts().sort_index())
display(merchants.head(10))


In [ ]:
# ------------------------------------------------------------
# 2. Aggregate Transaction & Fraud Metrics per Merchant
# ------------------------------------------------------------
merchant_stats = {}

if PREDICTIONS_PARQUET.exists():
    print(f"Aggregating directly from Model Predictions Parquet: {PREDICTIONS_PARQUET.name}...")
    pf = pq.ParquetFile(PREDICTIONS_PARQUET)
    for batch in pf.iter_batches(batch_size=250_000):
        df = batch.to_pandas()
        has_fraud = "is_fraud" in df.columns
        has_prob = "fraud_probability" in df.columns
        grouped = df.groupby("merchant_abn").agg(
            gmv=("dollar_value", "sum"),
            tx_count=("order_id", "count"),
            unique_users=("user_id", "nunique"),
            fraud_count=("is_fraud", "sum") if has_fraud else ("dollar_value", lambda x: 0),
            fraud_loss=("dollar_value", lambda x: x[df.loc[x.index, "is_fraud"].fillna(False)].sum()) if has_fraud else ("dollar_value", lambda x: 0.0),
            sum_fraud_prob=("fraud_probability", "sum") if has_prob else ("dollar_value", lambda x: 0.0)
        ).reset_index()
        
        for row in grouped.itertuples(index=False):
            abn = row.merchant_abn
            if abn not in merchant_stats:
                merchant_stats[abn] = {
                    "merchant_abn": abn,
                    "total_gmv": 0.0,
                    "total_transactions": 0,
                    "unique_users": 0,
                    "fraud_transactions": 0,
                    "fraud_loss": 0.0,
                    "sum_fraud_prob": 0.0
                }
            merchant_stats[abn]["total_gmv"] += float(row.gmv)
            merchant_stats[abn]["total_transactions"] += int(row.tx_count)
            merchant_stats[abn]["unique_users"] += int(row.unique_users)
            merchant_stats[abn]["fraud_transactions"] += int(row.fraud_count)
            merchant_stats[abn]["fraud_loss"] += float(row.fraud_loss)
            merchant_stats[abn]["sum_fraud_prob"] += float(row.sum_fraud_prob)
        del df, grouped
        gc.collect()

elif PREDICTIONS_CSV.exists():
    print(f"Aggregating from predictions CSV: {PREDICTIONS_CSV.name}...")
    chunks = pd.read_csv(PREDICTIONS_CSV, chunksize=250_000)
    for df in chunks:
        has_fraud = "is_fraud" in df.columns
        has_prob = "fraud_probability" in df.columns
        grouped = df.groupby("merchant_abn").agg(
            gmv=("dollar_value", "sum"),
            tx_count=("order_id", "count"),
            unique_users=("user_id", "nunique"),
            fraud_count=("is_fraud", "sum") if has_fraud else ("dollar_value", lambda x: 0),
            fraud_loss=("dollar_value", lambda x: x[df.loc[x.index, "is_fraud"].fillna(False)].sum()) if has_fraud else ("dollar_value", lambda x: 0.0),
            sum_fraud_prob=("fraud_probability", "sum") if has_prob else ("dollar_value", lambda x: 0.0)
        ).reset_index()
        
        for row in grouped.itertuples(index=False):
            abn = row.merchant_abn
            if abn not in merchant_stats:
                merchant_stats[abn] = {
                    "merchant_abn": abn,
                    "total_gmv": 0.0,
                    "total_transactions": 0,
                    "unique_users": 0,
                    "fraud_transactions": 0,
                    "fraud_loss": 0.0,
                    "sum_fraud_prob": 0.0
                }
            merchant_stats[abn]["total_gmv"] += float(row.gmv)
            merchant_stats[abn]["total_transactions"] += int(row.tx_count)
            merchant_stats[abn]["unique_users"] += int(row.unique_users)
            merchant_stats[abn]["fraud_transactions"] += int(row.fraud_count)
            merchant_stats[abn]["fraud_loss"] += float(row.fraud_loss)
            merchant_stats[abn]["sum_fraud_prob"] += float(row.sum_fraud_prob)
        del df, grouped
        gc.collect()

else:
    curated_files = sorted(CURATED_DIR.glob("*.parquet"))
    print(f"Aggregating from {len(curated_files)} curated parquet files in {CURATED_DIR}...")
    for f in curated_files:
        print(f" - Processing {f.name}...")
        pf = pq.ParquetFile(f)
        for batch in pf.iter_batches(batch_size=250_000):
            df = batch.to_pandas()
            has_fraud = "is_fraud" in df.columns
            has_prob = "fraud_probability" in df.columns
            grouped = df.groupby("merchant_abn").agg(
                gmv=("dollar_value", "sum"),
                tx_count=("order_id", "count"),
                unique_users=("user_id", "nunique"),
                fraud_count=("is_fraud", "sum") if has_fraud else ("dollar_value", lambda x: 0),
                fraud_loss=("dollar_value", lambda x: x[df.loc[x.index, "is_fraud"].fillna(False)].sum()) if has_fraud else ("dollar_value", lambda x: 0.0),
                sum_fraud_prob=("fraud_probability", "sum") if has_prob else ("dollar_value", lambda x: 0.0)
            ).reset_index()
            
            for row in grouped.itertuples(index=False):
                abn = row.merchant_abn
                if abn not in merchant_stats:
                    merchant_stats[abn] = {
                        "merchant_abn": abn,
                        "total_gmv": 0.0,
                        "total_transactions": 0,
                        "unique_users": 0,
                        "fraud_transactions": 0,
                        "fraud_loss": 0.0,
                        "sum_fraud_prob": 0.0
                    }
                merchant_stats[abn]["total_gmv"] += float(row.gmv)
                merchant_stats[abn]["total_transactions"] += int(row.tx_count)
                merchant_stats[abn]["unique_users"] += int(row.unique_users)
                merchant_stats[abn]["fraud_transactions"] += int(row.fraud_count)
                merchant_stats[abn]["fraud_loss"] += float(row.fraud_loss)
                merchant_stats[abn]["sum_fraud_prob"] += float(row.sum_fraud_prob)
            del df, grouped
            gc.collect()

# Convert dictionary to DataFrame
agg_df = pd.DataFrame.from_dict(merchant_stats, orient="index")
print(f"✓ Aggregated metrics successfully for {len(agg_df):,} merchants.")


In [ ]:
# ------------------------------------------------------------
# 3. Compute Financial & Risk Ratios
# ------------------------------------------------------------
merged_merchants = merchants.merge(agg_df, on="merchant_abn", how="inner")

# Average ticket size (AOV)
merged_merchants["avg_order_value"] = (
    merged_merchants["total_gmv"] / merged_merchants["total_transactions"].clip(lower=1)
).round(2)

# Fraud rate & mean fraud score
merged_merchants["fraud_rate"] = (
    merged_merchants["fraud_transactions"] / merged_merchants["total_transactions"].clip(lower=1)
).round(4)

merged_merchants["mean_fraud_prob"] = (
    merged_merchants["sum_fraud_prob"] / merged_merchants["total_transactions"].clip(lower=1)
).round(2)

# Net platform economics (3% standard take rate)
TAKE_RATE = 0.03
merged_merchants["estimated_platform_revenue"] = (merged_merchants["total_gmv"] * TAKE_RATE).round(2)
merged_merchants["net_platform_contribution"] = (
    merged_merchants["estimated_platform_revenue"] - merged_merchants["fraud_loss"]
).round(2)

# ------------------------------------------------------------
# 4. Compute Merchant Performance Index (MPI) - Segment & Overall
# ------------------------------------------------------------
def compute_mpi_score(df):
    """
    Calculate the Risk-Adjusted Merchant Performance Index (0 - 100):
    Scale (40%) + Reach (30%) + Ticket Size (15%) - Fraud Penalty (15%)
    """
    p_gmv = df["total_gmv"].rank(pct=True)
    p_users = df["unique_users"].rank(pct=True)
    p_aov = df["avg_order_value"].rank(pct=True)
    p_fraud = df["fraud_rate"].rank(pct=True)
    
    raw_score = (0.40 * p_gmv) + (0.30 * p_users) + (0.15 * p_aov) - (0.15 * p_fraud)
    score_min = raw_score.min()
    score_max = raw_score.max()
    return ((raw_score - score_min) / (score_max - score_min + 1e-6) * 100).round(2)

# 1. Segment-level MPI & Rank
segment_frames = []
for segment_name, group in merged_merchants.groupby("segment"):
    grp = group.copy()
    grp["segment_performance_score"] = compute_mpi_score(grp)
    grp["rank_in_segment"] = grp["segment_performance_score"].rank(ascending=False, method="min").astype(int)
    segment_frames.append(grp)

ranked_merchants = pd.concat(segment_frames, ignore_index=True)

# 2. Overall MPI & Rank across the entire merchant universe
ranked_merchants["overall_performance_score"] = compute_mpi_score(ranked_merchants)
ranked_merchants["overall_rank"] = (
    ranked_merchants["overall_performance_score"]
    .rank(ascending=False, method="min")
    .astype(int)
)

# Sort primarily by overall rank
ranked_merchants = ranked_merchants.sort_values("overall_rank").reset_index(drop=True)

print(f"✓ Completed ranking for {len(ranked_merchants):,} merchants across {ranked_merchants['segment'].nunique()} segments.")


In [ ]:
# ------------------------------------------------------------
# 5. Extract Top 10 per Segment & Top 100 Best Merchants Overall
# ------------------------------------------------------------
display_columns = [
    "overall_rank",
    "overall_performance_score",
    "name",
    "segment",
    "rank_in_segment",
    "segment_performance_score",
    "merchant_abn",
    "total_gmv",
    "total_transactions",
    "avg_order_value",
    "unique_users",
    "fraud_rate",
    "net_platform_contribution"
]

# 1. Top 100 Best Merchants Overall
top_100_overall = (
    ranked_merchants[ranked_merchants["overall_rank"] <= 100]
    .sort_values("overall_rank")
    .reset_index(drop=True)
)[display_columns]

# 2. Top 10 per Segment
top_10_per_segment = (
    ranked_merchants[ranked_merchants["rank_in_segment"] <= 10]
    .sort_values(["segment", "rank_in_segment"])
    .reset_index(drop=True)
)[display_columns]

# Save outputs
top_100_csv_path = TABLES_DIR / "top_100_merchants_overall.csv"
top_10_csv_path = TABLES_DIR / "top_10_merchants_per_segment.csv"
all_ranked_csv_path = TABLES_DIR / "all_merchants_segmented_ranked.csv"

top_100_overall.to_csv(top_100_csv_path, index=False)
top_10_per_segment.to_csv(top_10_csv_path, index=False)
ranked_merchants.to_csv(all_ranked_csv_path, index=False)

print(f"✓ Saved Top 100 Overall ({len(top_100_overall)} rows) to: {top_100_csv_path}")
print(f"✓ Saved Top 10 per segment ({len(top_10_per_segment)} rows) to: {top_10_csv_path}")
print(f"✓ Saved full ranked dataset ({len(ranked_merchants)} rows) to: {all_ranked_csv_path}")


In [ ]:
# ------------------------------------------------------------
# 6. Display Top 100 Overall & Segment Breakdown
# ------------------------------------------------------------
print("=" * 95)
print("🏆 TOP 20 PREVIEW OF THE TOP 100 BEST MERCHANTS OVERALL")
print("=" * 95)
display(top_100_overall.head(20))

# Segment Distribution in Top 100
top_100_segment_dist = top_100_overall["segment"].value_counts().reset_index()
top_100_segment_dist.columns = ["Segment", "Merchant Count in Top 100"]
top_100_segment_dist["Percentage (%)"] = (top_100_segment_dist["Merchant Count in Top 100"] / len(top_100_overall) * 100).round(1)

print("\n" + "=" * 95)
print("📊 TOP 100 MERCHANTS: RETAIL SEGMENT DISTRIBUTION")
print("=" * 95)
display(top_100_segment_dist)

# Display Top 10 per Segment
for seg in top_10_per_segment["segment"].unique():
    print("\n" + "=" * 95)
    print(f"🏆 TOP 10 MERCHANTS: {seg.upper()}")
    print("=" * 95)
    sub = top_10_per_segment[top_10_per_segment["segment"] == seg].drop(columns=["segment"])
    display(sub)


In [ ]:
# ------------------------------------------------------------
# 7. Visualizations: Top 100 Overall Portfolio Insights
# ------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 1. Segment breakdown of Top 100 merchants
sns.barplot(
    data=top_100_segment_dist,
    y="Segment",
    x="Merchant Count in Top 100",
    palette="viridis",
    ax=axes[0]
)
axes[0].set_title("Top 100 Merchants: Composition by Industry Segment", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Number of Merchants in Top 100")
axes[0].grid(axis="x", linestyle="--", alpha=0.5)

# 2. GMV vs Fraud Rate for Top 100 Merchants
sns.scatterplot(
    data=top_100_overall,
    x="total_gmv",
    y="fraud_rate",
    hue="segment",
    size="unique_users",
    sizes=(60, 300),
    alpha=0.85,
    ax=axes[1]
)
axes[1].set_title("Top 100 Merchants: GMV vs. Fraud Rate", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Total GMV ($)")
axes[1].set_ylabel("Fraud Rate (Proportion)")
axes[1].grid(True, linestyle="--", alpha=0.5)
axes[1].legend(bbox_to_anchor=(1.02, 1), loc="upper left", borderaxespad=0.)

plt.tight_layout()
plt.show()
